<a href="https://colab.research.google.com/github/gns1719/Pet-NosePrint-Id-Service/blob/Jun/Triplet_test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np
from skimage.feature import local_binary_pattern
import os
from google.colab import files
import io

# --- Triplet Network 정의 ---
class TripletNetwork(nn.Module):
    def __init__(self):
        super(TripletNetwork, self).__init__()
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.fc = nn.Identity()  # 마지막 FC 제거
        self.embedding = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)  # 출력 임베딩 차원
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.embedding(x)

# --- LBP 전처리 함수 ---
def preprocess_lbp_image(image_path):
    image = Image.open(image_path).convert('L')
    image_np = np.array(image)
    lbp = local_binary_pattern(image_np, P=8, R=1, method='uniform')
    lbp = (lbp / lbp.max() * 255).astype(np.uint8)
    lbp_img = Image.fromarray(lbp).convert('RGB')

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    return transform(lbp_img).unsqueeze(0)

# --- 모델 로드 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TripletNetwork().to(device)

# 📌 모델 파일 업로드
print("🔼 모델 파일(.pth)을 업로드하세요.")
uploaded = files.upload()

for fn in uploaded.keys():
    if fn.endswith('.pth'):
        model.load_state_dict(torch.load(io.BytesIO(uploaded[fn]), map_location=device))
        print(f"✅ 모델 로드 완료: {fn}")
        break

model.eval()

# --- 비교 함수 ---
def compare_images(anchor_path, compare_path, threshold=5.0):
    anchor = preprocess_lbp_image(anchor_path).to(device)
    compare = preprocess_lbp_image(compare_path).to(device)

    with torch.no_grad():
        anchor_emb = model(anchor)
        compare_emb = model(compare)
        distance = F.pairwise_distance(anchor_emb, compare_emb).item()

    print(f"📏 유클리디안 거리 (Anchor vs. Compare): {distance:.4f}")
    if distance < threshold:
        print("✅ 두 이미지는 유사합니다.")
    else:
        print("❌ 두 이미지는 다릅니다.")

# --- 이미지 업로드 및 비교 예시 ---
print("\n🔼 Anchor 이미지 업로드")
anchor_file = files.upload()
anchor_path = list(anchor_file.keys())[0]

print("\n🔼 비교할 이미지 업로드")
compare_file = files.upload()
compare_path = list(compare_file.keys())[0]

compare_images(anchor_path, compare_path, threshold=5.0)


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 153MB/s]


🔼 모델 파일(.pth)을 업로드하세요.


Saving triplet_lbp_model.pth to triplet_lbp_model.pth
✅ 모델 로드 완료: triplet_lbp_model.pth

🔼 Anchor 이미지 업로드


Saving coco1.jpg to coco1.jpg

🔼 비교할 이미지 업로드


Saving coco2.jpg to coco2.jpg
📏 유클리디안 거리 (Anchor vs. Compare): 2.7407
✅ 두 이미지는 유사합니다.


In [ ]:
from google.colab import files
uploaded_files = files.upload()

Saving triplet_lbp_model.pth to triplet_lbp_model.pth


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np
from skimage.feature import local_binary_pattern
import os

# --- Triplet Network 정의 ---
class TripletNetwork(nn.Module):
    def __init__(self):
        super(TripletNetwork, self).__init__()
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.fc = nn.Identity()  # 마지막 FC 제거
        self.embedding = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)  # 출력 임베딩 차원
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.embedding(x)

# --- LBP 전처리 함수 ---
def preprocess_lbp_image(image_path):
    # 흑백으로 열기
    image = Image.open(image_path).convert('L')
    image_np = np.array(image)

    # LBP 적용
    lbp = local_binary_pattern(image_np, P=8, R=1, method='uniform')
    lbp = (lbp / lbp.max() * 255).astype(np.uint8)
    lbp_img = Image.fromarray(lbp).convert('RGB')

    # transform
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    return transform(lbp_img).unsqueeze(0)  # 배치 차원 추가

# --- 디바이스 및 모델 로드 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TripletNetwork().to(device)
model.load_state_dict(torch.load('triplet_lbp_model.pth', map_location=device))
model.eval()

# --- 단일 비교 함수 (Anchor vs 비교 대상) ---
def compare_images(anchor_path, compare_path, threshold=10.0):
    anchor = preprocess_lbp_image(anchor_path).to(device)
    compare = preprocess_lbp_image(compare_path).to(device)

    with torch.no_grad():
        anchor_emb = model(anchor)
        compare_emb = model(compare)
        distance = F.pairwise_distance(anchor_emb, compare_emb).item()

    print(f"📏 유클리디안 거리 (Anchor vs. Compare): {distance:.4f}")
    if distance < threshold:
        print("✅ 두 이미지는 유사합니다.")
    else:
        print("❌ 두 이미지는 다릅니다.")

# --- 여러 이미지와 비교 함수 ---
def compare_with_multiple(anchor_path, compare_paths, threshold=0.7):
    anchor = preprocess_lbp_image(anchor_path).to(device)

    with torch.no_grad():
        anchor_emb = model(anchor)

        print(f"\n📌 Anchor: {anchor_path}")
        for path in compare_paths:
            compare = preprocess_lbp_image(path).to(device)
            compare_emb = model(compare)
            distance = F.pairwise_distance(anchor_emb, compare_emb).item()
            print(f" - [{os.path.basename(path)}] 거리: {distance:.4f} → {'✅ 유사' if distance < threshold else '❌ 다름'}")

# --- 사용 예시 ---
# compare_images('/path/to/anchor.jpg', '/path/to/other.jpg')

# compare_with_multiple('/path/to/anchor.jpg', [
#     '/path/to/other1.jpg',
#     '/path/to/other2.jpg',
#     '/path/to/other3.jpg'
# ])


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 151MB/s]


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np
from skimage.feature import local_binary_pattern
import os

# --- Triplet Network 정의 ---
class TripletNetwork(nn.Module):
    def __init__(self):
        super(TripletNetwork, self).__init__()
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.fc = nn.Identity()  # 마지막 FC 제거
        self.embedding = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)  # 출력 임베딩 차원
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.embedding(x)

# --- LBP 전처리 함수 ---
def preprocess_lbp_image(image_path):
    # 흑백으로 열기
    image = Image.open(image_path).convert('L')
    image_np = np.array(image)

    # LBP 적용
    lbp = local_binary_pattern(image_np, P=8, R=1, method='uniform')
    lbp = (lbp / lbp.max() * 255).astype(np.uint8)
    lbp_img = Image.fromarray(lbp).convert('RGB')

    # transform
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    return transform(lbp_img).unsqueeze(0)  # 배치 차원 추가

# --- 디바이스 및 모델 로드 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TripletNetwork().to(device)
model.load_state_dict(torch.load('triplet_lbp_model.pth', map_location=device))
model.eval()

# --- 단일 비교 함수 (Anchor vs 비교 대상) ---
def compare_images(anchor_path, compare_path, threshold=10.0, cosine_threshold=0.8):
    anchor = preprocess_lbp_image(anchor_path).to(device)
    compare = preprocess_lbp_image(compare_path).to(device)

    with torch.no_grad():
        anchor_emb = model(anchor)
        compare_emb = model(compare)
        euclidean = F.pairwise_distance(anchor_emb, compare_emb).item()
        cosine = F.cosine_similarity(anchor_emb, compare_emb).item()

    print(f"\n📏 유클리디안 거리: {euclidean:.4f}")
    print(f"📐 코사인 유사도: {cosine:.4f}")
    if euclidean < threshold and cosine > cosine_threshold:
        print("✅ 두 이미지는 유사합니다.")
    else:
        print("❌ 두 이미지는 다릅니다.")

# --- 여러 이미지와 비교 함수 ---
def compare_with_multiple(anchor_path, compare_paths, threshold=10.0, cosine_threshold=0.8):
    anchor = preprocess_lbp_image(anchor_path).to(device)

    with torch.no_grad():
        anchor_emb = model(anchor)

        print(f"\n📌 Anchor: {anchor_path}")
        for path in compare_paths:
            compare = preprocess_lbp_image(path).to(device)
            compare_emb = model(compare)
            euclidean = F.pairwise_distance(anchor_emb, compare_emb).item()
            cosine = F.cosine_similarity(anchor_emb, compare_emb).item()
            result = '✅ 유사' if euclidean < threshold and cosine > cosine_threshold else '❌ 다름'
            print(f" - [{os.path.basename(path)}] 거리: {euclidean:.4f}, 코사인 유사도: {cosine:.4f} → {result}")

# --- 사용 예시 ---
# compare_images('/path/to/anchor.jpg', '/path/to/other.jpg')

# compare_with_multiple('/path/to/anchor.jpg', [
#     '/path/to/other1.jpg',
#     '/path/to/other2.jpg',
#     '/path/to/other3.jpg'
# ])


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 166MB/s]


In [ ]:
# 비교할 이미지 경로 설정
image1_path = '76.jpg'
image2_path = '80.jpg'

# 비교 실행
compare_images(image1_path, image2_path)


📏 유클리디안 거리: 13.3518
📐 코사인 유사도: 0.8810
❌ 두 이미지는 다릅니다.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models, transforms
from PIL import Image
import numpy as np
from skimage.feature import local_binary_pattern
import os

# --- Triplet Network 정의 ---
class TripletNetwork(nn.Module):
    def __init__(self):
        super(TripletNetwork, self).__init__()
        self.backbone = models.resnet18(pretrained=True)
        self.backbone.fc = nn.Identity()  # 마지막 FC 제거
        self.embedding = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)  # 출력 임베딩 차원
        )

    def forward(self, x):
        x = self.backbone(x)
        return self.embedding(x)

# --- LBP 전처리 함수 ---
def preprocess_lbp_image(image_path):
    # 1. 흑백으로 열기
    image = Image.open(image_path).convert('L')
    image_np = np.array(image)

    # 2. 마스크 생성 (검은 배경 제거용)
    threshold = 15  # 픽셀 값 0~15는 거의 검은색으로 간주
    mask = image_np > threshold

    # 3. 마스크가 있는 부분만 바운딩 박스 추출
    coords = np.argwhere(mask)
    if coords.size == 0:
        raise ValueError(f"No foreground found in {image_path}")
    y0, x0 = coords.min(axis=0)
    y1, x1 = coords.max(axis=0) + 1  # 슬라이싱을 위해 +1

    cropped = image_np[y0:y1, x0:x1]

    # 4. 잘라낸 영역을 정사각형 중앙에 배치 (패딩)
    h, w = cropped.shape
    max_side = max(h, w)
    square = np.zeros((max_side, max_side), dtype=np.uint8)
    top = (max_side - h) // 2
    left = (max_side - w) // 2
    square[top:top+h, left:left+w] = cropped

    # 5. LBP 적용
    lbp = local_binary_pattern(square, P=8, R=1, method='uniform')
    lbp = (lbp / lbp.max() * 255).astype(np.uint8)
    lbp_img = Image.fromarray(lbp).convert('RGB')

    # 6. Transform 적용
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.5]*3, [0.5]*3)
    ])
    return transform(lbp_img).unsqueeze(0)


# --- 디바이스 및 모델 로드 ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = TripletNetwork().to(device)
model.load_state_dict(torch.load('triplet_lbp_model.pth', map_location=device))
model.eval()

# --- 1:1 비교 함수 ---
def compare_images(anchor_path, compare_path, threshold=10.0, cosine_threshold=0.8):
    anchor = preprocess_lbp_image(anchor_path).to(device)
    compare = preprocess_lbp_image(compare_path).to(device)

    with torch.no_grad():
        anchor_emb = model(anchor)
        compare_emb = model(compare)
        euclidean = F.pairwise_distance(anchor_emb, compare_emb).item()
        cosine = F.cosine_similarity(anchor_emb, compare_emb).item()

    print(f"\n📏 유클리디안 거리: {euclidean:.4f}")
    print(f"📐 코사인 유사도: {cosine:.4f}")
    if euclidean < threshold and cosine > cosine_threshold:
        print("✅ 두 이미지는 유사합니다.")
    else:
        print("❌ 두 이미지는 다릅니다.")

# --- 1:N 비교 함수 ---
def compare_with_multiple(anchor_path, compare_paths, threshold=10.0, cosine_threshold=0.8):
    anchor = preprocess_lbp_image(anchor_path).to(device)

    with torch.no_grad():
        anchor_emb = model(anchor)

        print(f"\n📌 Anchor: {anchor_path}")
        for path in compare_paths:
            compare = preprocess_lbp_image(path).to(device)
            compare_emb = model(compare)
            euclidean = F.pairwise_distance(anchor_emb, compare_emb).item()
            cosine = F.cosine_similarity(anchor_emb, compare_emb).item()
            result = '✅ 유사' if euclidean < threshold and cosine > cosine_threshold else '❌ 다름'
            print(f" - [{os.path.basename(path)}] 거리: {euclidean:.4f}, 코사인 유사도: {cosine:.4f} → {result}")

# --- N:1 비교 함수 (여러 Anchor vs 하나의 대상) ---
def compare_multiple_anchors_to_one(anchor_paths, compare_path, threshold=10.0, cosine_threshold=0.8):
    anchors = [preprocess_lbp_image(p).to(device) for p in anchor_paths]
    compare = preprocess_lbp_image(compare_path).to(device)

    with torch.no_grad():
        # Anchor들의 임베딩 계산 및 평균
        anchor_embeddings = torch.cat([model(a) for a in anchors], dim=0)  # [N, 128]
        anchor_avg = torch.mean(anchor_embeddings, dim=0, keepdim=True)   # [1, 128]

        # 비교 대상 임베딩
        compare_emb = model(compare)

        # 평균 anchor vs 비교 대상 거리 및 유사도
        euclidean = F.pairwise_distance(anchor_avg, compare_emb).item()
        cosine = F.cosine_similarity(anchor_avg, compare_emb).item()

    print(f"\n📌 Anchors: {[os.path.basename(p) for p in anchor_paths]}")
    print(f"🔍 Compare: {os.path.basename(compare_path)}")
    print(f"📏 평균 유클리디안 거리: {euclidean:.4f}")
    print(f"📐 평균 코사인 유사도: {cosine:.4f}")

    if euclidean < threshold and cosine > cosine_threshold:
        print("✅ 평균 Anchor 기준으로 유사합니다.")
    else:
        print("❌ 평균 Anchor 기준으로 다릅니다.")


def compare_multiple_anchors_to_two(anchor_paths, compare_path, threshold=10.0, cosine_threshold=0.8, positive_count_threshold=1):
    anchors = [preprocess_lbp_image(p).to(device) for p in anchor_paths]
    compare = preprocess_lbp_image(compare_path).to(device)

    with torch.no_grad():
        compare_emb = model(compare)

        positive_results = []
        cosine_similarities = []

        print(f"\n📌 Anchors: {[os.path.basename(p) for p in anchor_paths]}")
        print(f"🔍 Compare: {os.path.basename(compare_path)}")

        for i, a in enumerate(anchors):
            anchor_emb = model(a)
            euclidean = F.pairwise_distance(anchor_emb, compare_emb).item()
            cosine = F.cosine_similarity(anchor_emb, compare_emb).item()
            cosine_similarities.append(cosine)
            is_similar = (euclidean < threshold and cosine > cosine_threshold)
            positive_results.append(is_similar)
            print(f" - Anchor[{i+1}] 거리: {euclidean:.4f}, 코사인 유사도: {cosine:.4f} → {'✅ 유사' if is_similar else '❌ 다름'}")

        positive_count = sum(positive_results)
        max_cosine = max(cosine_similarities)

        print(f"\n✅ 유사한 Anchor 이미지 수: {positive_count} / {len(anchors)}")
        print(f"🔝 최대 코사인 유사도: {max_cosine:.4f}")

        if positive_count >= positive_count_threshold:
            print(f"🎯 최종 판정: ✅ 유사합니다. (유사 이미지 {positive_count}장 이상)")
        else:
            print(f"🎯 최종 판정: ❌ 다릅니다. (유사 이미지 {positive_count}장 미만)")

# --- 사용 예시 ---
# 1:1 비교
# compare_images('data/anchor.jpg', 'data/target.jpg')

# 1:N 비교
# compare_with_multiple('data/anchor.jpg', [
#     'data/target1.jpg',
#     'data/target2.jpg'
# ])

# N:1 비교
# compare_multiple_anchors_to_one([
#     'data/anchor1.jpg',
#     'data/anchor2.jpg'
# ], 'data/target.jpg')


/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
<ipython-input-28-6e22478f66d3>:70: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more detai

In [ ]:
# 여러 장의 Anchor 이미지 경로 설정
anchor_paths = ['35.jpg', '36.jpg']

# 비교 대상 이미지 경로 설정 (새로 찍은 이미지)
compare_path = 'coco2.jpg'

# 비교 실행
compare_multiple_anchors_to_one(anchor_paths, compare_path)
compare_multiple_anchors_to_two(anchor_paths, compare_path)


📌 Anchors: ['35.jpg', '36.jpg']
🔍 Compare: coco2.jpg
📏 평균 유클리디안 거리: 56.1295
📐 평균 코사인 유사도: 0.4247
❌ 평균 Anchor 기준으로 다릅니다.

📌 Anchors: ['35.jpg', '36.jpg']
🔍 Compare: coco2.jpg
 - Anchor[1] 거리: 56.0142, 코사인 유사도: 0.4297 → ❌ 다름
 - Anchor[2] 거리: 56.2775, 코사인 유사도: 0.4190 → ❌ 다름

✅ 유사한 Anchor 이미지 수: 0 / 2
🔝 최대 코사인 유사도: 0.4297
🎯 최종 판정: ❌ 다릅니다. (유사 이미지 0장 미만)
